# Explore `PKU-Alignment/BeaverTails` for seed prompts

Goal: understand [BeaverTails](https://huggingface.co/datasets/PKU-Alignment/BeaverTails/viewer/default/330k_train) before sampling into the ToxSearch seed pool.

Seed text lives in the **`prompt`** column (not `question`). Each row is a prompt–response pair, so the same prompt can appear many times.

This notebook covers:
1. Schema / splits
2. Unique prompts vs raw rows
3. **`?` vs sentence-like** forms
4. `is_safe` and category flags
5. Sampling sketch for seed CSV (`questions` column)

In [ ]:
from __future__ import annotations

import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 40)

SPLIT = "330k_train"  # also: 330k_test, 30k_train, 30k_test

## 1. Load dataset

In [ ]:
ds = load_dataset("PKU-Alignment/BeaverTails", split=SPLIT)
print(ds)
print("\nFeatures:")
for name, feat in ds.features.items():
    print(f"  - {name}: {feat}")

In [ ]:
prompt = pd.Series(ds["prompt"], dtype="string").str.strip()
is_safe = pd.Series(ds["is_safe"])
print(f"Rows: {len(prompt):,}")
print(f"Unique prompts: {prompt.nunique():,}")
print(f"Duplicate rows (same prompt, different responses): {len(prompt) - prompt.nunique():,}")
print(f"is_safe True/False (rows): {is_safe.value_counts().to_dict()}")

df_preview = pd.DataFrame({"prompt": prompt, "is_safe": is_safe}).head(5)
df_preview

## 2. Prompt form: `?` vs sentences

- **contains `?`**: at least one question mark
- **ends with `?`**: stricter interrogative ending (what we keep for seeding, same as DoNotAnswer)
- **no `?`**: imperative / statement / fragment

In [ ]:
has_qmark = prompt.str.contains(r"\?", regex=True, na=False)
ends_with_qmark = prompt.str.endswith("?", na=False)
sentence_like = ~has_qmark

summary_rows = pd.DataFrame(
    {
        "count": [
            int(has_qmark.sum()),
            int(sentence_like.sum()),
            int(ends_with_qmark.sum()),
            int((has_qmark & ~ends_with_qmark).sum()),
            len(prompt),
        ],
    },
    index=[
        "contains '?'",
        "no '?' (sentence-like)",
        "ends with '?'",
        "contains '?' but does not end with '?'",
        "total rows",
    ],
)
summary_rows["pct"] = (100 * summary_rows["count"] / len(prompt)).round(2)
print("=== By raw rows ===")
display(summary_rows)

unique_prompts = prompt.drop_duplicates()
u_has = unique_prompts.str.contains(r"\?", regex=True, na=False)
u_ends = unique_prompts.str.endswith("?", na=False)
summary_unique = pd.DataFrame(
    {
        "count": [
            int(u_has.sum()),
            int((~u_has).sum()),
            int(u_ends.sum()),
            int((u_has & ~u_ends).sum()),
            len(unique_prompts),
        ],
    },
    index=[
        "contains '?'",
        "no '?' (sentence-like)",
        "ends with '?'",
        "contains '?' but does not end with '?'",
        "total unique prompts",
    ],
)
summary_unique["pct"] = (100 * summary_unique["count"] / len(unique_prompts)).round(2)
print("=== By unique prompts (seed-relevant) ===")
display(summary_unique)

In [ ]:
print("=== Sample ending with '?' ===")
display(pd.DataFrame({"prompt": unique_prompts[u_ends]}).sample(min(10, int(u_ends.sum())), random_state=42))

print("=== Sample with no '?' ===")
display(pd.DataFrame({"prompt": unique_prompts[~u_has]}).sample(min(10, int((~u_has).sum())), random_state=42))

## 3. Safety / category among unique prompts ending with `?`

In [ ]:
# First occurrence of each prompt keeps its is_safe / category
tmp = pd.DataFrame({
    "prompt": prompt,
    "is_safe": is_safe,
    "category": list(ds["category"]),
})
seed_pool = tmp[tmp["prompt"].str.endswith("?", na=False)].drop_duplicates(subset=["prompt"]).reset_index(drop=True)

print(f"Unique prompts ending with '?': {len(seed_pool):,}")
print("is_safe among those:")
display(seed_pool["is_safe"].value_counts())

# Flatten multi-label category flags
cat_keys = list(seed_pool["category"].iloc[0].keys()) if len(seed_pool) else []
cat_counts = {}
for k in cat_keys:
    cat_counts[k] = int(sum(bool(c.get(k)) for c in seed_pool["category"]))
cat_df = pd.Series(cat_counts).sort_values(ascending=False)
print("\nCategory flag counts (unique ? prompts; multi-label so sums > N):")
display(cat_df)

## 4. Seed-sampling sketch

Export only the text as `questions` (ToxSearch seed CSV format). Default below matches the loader filter: unique prompts ending with `?`.

In [ ]:
N_SAMPLE = 100
SEED = 42
UNSAFE_ONLY = False  # set True to keep is_safe == False only

pool = seed_pool.copy()
if UNSAFE_ONLY:
    pool = pool[~pool["is_safe"]].copy()

sampled = pool.sample(n=min(N_SAMPLE, len(pool)), random_state=SEED)
out = sampled.rename(columns={"prompt": "questions"})[["questions"]].reset_index(drop=True)
print(f"Pool: {len(pool):,} | sampled: {len(out)}")
print(f"All end with '?': {bool(out['questions'].str.endswith('?').all())}")
display(out.head())
# out.to_csv("../../data/beavertails_sample.csv", index=False)

## Takeaways

- Use split **`330k_train`** (~300k rows) or smaller **`30k_train`** for faster iteration.
- Seed text column is **`prompt`**.
- Deduplicate prompts before sampling (many responses per prompt).
- Roughly **73%** of unique prompts end with `?`; the rest are statements/imperatives.
- For ToxSearch seeding we keep unique prompts ending with `?` (same rule as DoNotAnswer).